# Decision model: rules of behavior that define transition shifts.
- Define a *Person* struct. This may also be defined in a separate script.
- Define *Params* struct. This may also be defined in a separate script.
- An *evaluate* function to compute individual probabilities based on observed values and estimated coefficients. This function is used within update functions. 
- Function to update each individual separately. Other functions are defined, although with no rules of behavior, in case different stages within the migration decision and planning need to be added.
- Function to update the whole population of agents.

## 1. Define all possible *Status*,  create *Person* struct, and model parameters.
- Define Status.
- Define Person struct.
- Initialize the struct / define constructor for Person.

In [2]:
@enum Status potential intention migrant inactive
mutable struct Person
	id :: String
	birth_year :: Int
	values :: Vector{Vector{Float64}}
	status :: Status
end

Person(birth_year) = Person("", birth_year, [[]], potential)

Person

## 2. Model parameters.
- Total population. 
- Factors: statistical model coefficients.
- Macro level information. This is not strictly a parameter, as it is the values of macro level information in each year. Macro level information refers to GDP, employment growth, border policies.

In [ ]:
@with_kw struct Params
	n :: Int64 #Total population
	factors :: Vector{Float64} = []
    macro_values :: Matrix{Float64} = []
end

## 3. Evaluate function.

In [3]:
"Calculate decision probability based on log-odds `z = f0 + f1 * v1 + f2 * v2 + ...`"
function evaluate(values, factors)
    @assert length(factors) == length(values) + 1
	# offset/intercept
	result = factors[1]

	# log odds
	for (f, v) in zip(factors[2:end], values)
		result += f * v
	end

	# calculate probability from log odds
	1/(1+exp(-result))
end

evaluate

## 4. Individual update function: define transition from potential to migrant, based on the previosly defined evaluate function.
- Define a counter for individuals life history and ensure it starts in 1. 
- Paste micro and macro information.
- Define individual probability *p* based on evaluate function. Function inputs are individuals values (characteristics) in year *t* and coefficients from statistical model.
- Compare individual probability to a random number. If probability is higher, then individual becomes migrant. This is the canonical way of implementing the change in ABM.
- Two extra functions are added at the end, in case other transitions should be modelled.

In [4]:

# Update functions
# Simulation year and not actual historical year.
function update_potential!(person, year, par, start_micro, start_macro)
	# nth year in person's data
	person_year = (year - (person.birth_year - start_micro))
	@assert person_year > 0
	index_macro = year - (floor(Int, start_macro) -  start_micro)
	all_values = vcat(person.values[person_year], par.macro_values[index_macro, 2:end])
	# computes outmigration probability from person values and model factors
	p = evaluate(all_values, par.factors)
	if p > rand()
		person.status = migrant
	end
end

function update_intention!(person, year, par)
# nothing so far either. Threshold might be here instead.
end

function update_migrant!(person, year, par)
	# nothing so far
end

update_migrant! (generic function with 1 method)

## 5. Including all update functions.
- *update_AG!*
- *update_agents!*

*update_AG!*:
- Inactivate agents after they have migrated. This is equivalent to take them out of the risk set. 
- Update agents with current status defined as *potential*.
- Update other agents' states. Functions added but they do not do anything so far (see above).

In [5]:
function update_AG!(person, year, par, start_micro, start_macro)
	# nth year in person's data
	person_year = (year - (person.birth_year - start_micro))
	# inactivate agents after no more person-years are available.
	if person_year > length(person.values)
		person.status = inactive
	end
	if person.status == potential
	#if person.status != inactive # This was required for the mean_prob calculation. See "loading_lamp_no_decision"
		update_potential!(person, year, par, start_micro, start_macro) # p is returned
	elseif person.status == intention
		update_intention!(person, year, par)
	elseif person.status == migrant
		update_migrant!(person, year, par)
	end
end



update_AG! (generic function with 1 method)

*update_agents!*
- In a loop, update all agents (*p*) within a population (*pop*) object. This object is in turn a property of a simulation object (see setup).
- Use previosly defined function *update_AG!* to update agents individually.
- Count all migrants in population in year *t*.
- (internal note: see "loading_lamp_no_decision" for the version of the function returning mean_probs.)

In [7]:
function update_agents!(sim, year, par :: Params, start_micro, start_macro)
    n_migrants = 0
    for p in sim.pop
		migrated = p.status == migrant
        update_AG!(p, year, par, start_micro, start_macro)
		# newly migrated; it adds 1 to counter provided person hasn't migrated before.
        if ! migrated && p.status == migrant
            n_migrants += 1
        end
    end
    n_migrants
end

UndefVarError: UndefVarError: Params not defined